In [1]:
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
load_dotenv()

True

#tools binding, native tool by lang chain

In [2]:
#tool-1
from langchain_community.tools import DuckDuckGoSearchRun
search=DuckDuckGoSearchRun()
search.invoke("obama's mother name?")
#this is the same as we do google search
#this is one tool



/var/folders/2z/pqpq9wcs0dv4jfbgkp96j3780000gn/T/ipykernel_5683/3272911525.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


"Michelle LaVaughn Robinson Obama[1] (née Robinson; born January 17, 1964) is an American attorney and author who served as First Lady of the United States from 2009 to 2017 as the wife of Barack Obama, the 44th president of the United States. Stanley Ann Dunham was the mother of Barack Obama, and she died more than a decade before he became president of the United States. She was an anthropologist who raised her two biracial children as a single mother while pursuing her dreams of helping to create sustainable business models for women in rural Indonesian villages. Barack Obama's parents married while students at the University of Hawaii. His father, Barack Obama, Sr., a Kenyan, became an economist in the government of Kenya. His mother, S. Ann Dunham, became an anthropologist. They divorced in 1964. Ann then married (and later divorced) another foreign student, Indonesian Lolo Soetoro. Barack Obama, the 44th President of the United States, entered the world on August 4, 1961, in Hono

In [6]:
#tool-2
#so now issue is that who gtp will know , this is the information and i have to go this tool. for this 
#we create the custome tool and bind  all tools in that and guide tool

#from langchain_community.tools import  WikipediaQueryRun
#from langchain_community.utilities import WikipediaAPIWrapper

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

#wikipedia = WikipediaQueryRun (api_wrapper=WikipediaAPIWrapper ())
wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

wikipedia.invoke("Sam Altman")

#print(wikipedia.invoke("Sam Altman"))

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [7]:
from langchain.tools import tool

In [10]:
@tool
def tool_duck_duck_go_search(query:str) -> str:
    """Use this tool when you need to answer questions about current events or general knowledge. """
    #tool-A
    from langchain_community.tools import DuckDuckGoSearchRun
    search=DuckDuckGoSearchRun()
    response=search.invoke(query)
    return response

In [11]:
tool_duck_duck_go_search.invoke("What is the capital of France?")

'4 days ago ... Paris is the capital and largest city of France, with an estimated city population of 2.04 million in an area of 105.4 km 2 (40.7 sq mi), and a metropolitan ... 7 Apr 2026 ... What is the capital of FRANCE? 108K. Dislike. 4,359. Share. Video unavailable. This content isn\'t available. Skip video. 27 Nov 2025 ... PARIS it is the capital and largest city of France. Located on the Seine in the country\'s north, it is a major cultural and political centre of Europe. 3 Dec 2025 ... The correct answer is Paris, which is the capital and largest city of France. This beautiful city, often called the "City of Light," has served as France\'s ... 21 Mar 2026 ... The correct answer is Paris. Paris is the capital of France. Key Points. The President of France is Emmanuel Macron. Paris is a city, and the capital of ...'

In [12]:
@tool
def tool_wiki_pedia_search(query: str) -> str:
    #tool-B

    """Use this tool when you need to answer questions about persons, places, etc. """

    from langchain_community.tools import  WikipediaQueryRun
    from langchain_community.utilities import WikipediaAPIWrapper

    wikipedia = WikipediaQueryRun (api_wrapper=WikipediaAPIWrapper ())
    res_wiki=wikipedia.invoke(query)
    return res_wiki

In [13]:
tool_wiki_pedia_search.invoke("Sam Altman")

'Page: Sam Altman\nSummary: Samuel Harris Altman (born April 22, 1985) is an American entrepreneur and investor who has been the chief executive officer (CEO) of the artificial intelligence company OpenAI since 2019.\nAltman attended Stanford University for two years before he dropped out and co-founded Loopt, a smartphone geosocial networking service. Loopt was acquired by Green Dot Corporation for $43.4 million. In 2011, Altman joined Y Combinator, a technology startup accelerator and venture capital firm, and was the company\'s president from 2014 to 2019. He is a billionaire with many investments including Reddit, Worldcoin, and Helion Energy.\nAltman co-founded OpenAI in 2015 and became its CEO in 2019, a role that made him a prominent figure of the AI boom. He supervised the launch of ChatGPT in November 2022. In 2023, he was ousted by the organization\'s board of directors for not being "consistently candid". The move was met with significant backlash from employees and investor

In [14]:
from langchain_core.tools import tool
import arxiv

@tool
def tool_arxiv_search(query: str, max_results: int = 3) -> list:
    #tool-C
    """Use this tool when you need to answer questions about scientific papers or research topics."""

    client = arxiv.Client()

    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.SubmittedDate
    )

    results = client.results(search)

    papers = []
    for r in results:
        papers.append({
            "title": r.title,
            "published": str(r.published),
            "summary": r.summary[:300],
            "link": r.entry_id
        })

    return papers

In [15]:
tool_arxiv_search.invoke({"query": "deep learning natural language processing", "max_results": 2})

HTTPError: Page request resulted in HTTP 301: None (http://export.arxiv.org/api/query?search_query=deep+learning+natural+language+processing&id_list=&sortBy=submittedDate&sortOrder=descending&start=0&max_results=2)

In [16]:

@tool
def tool_arxiv_search(query: str) -> str:
    #tool-C-1
    """Use this tool when you need to answer questions about scientific papers or research topics. """

    from langchain_community.tools import ArxivQueryRun
    from langchain_community.utilities import ArxivAPIWrapper

    # 1. Initialize the arXiv API wrapper
    arxiv_wrapper = ArxivAPIWrapper(
        top_k_results=3,       # Number of papers to retrieve
        doc_content_chars_max=2000  # Max characters per document
    )

    # 2. Create the arXiv tool
    arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)

    # 3. Use the tool directly
    result = arxiv_tool.run(query)

    print(result)

In [21]:
@tool
def tool_personal_info2(name: str) -> str:
    #tool-C-2
    """Use this tool when you need to answer questions about personal information.
    Args:
        name (str): The name of the person to look up.
    Returns:
        str: A string containing the person's age and occupation, or a message if the information is not found.
    """
    
    infos = [{
        "name": "John Doe",
        "age": 30,
        "occupation": "Software Engineer"
    },
    {
        "name": "Jane Smith",
        "age": 25,
        "occupation": "Data Scientist"
    }]

    for info in infos:
        if info["name"].lower() == name.lower():
            return f"{info['name']} is {info['age']} years old and works as a {info['occupation']}."
    return "Information not found."

In [22]:
tool_personal_info2.invoke("John Doe")

'John Doe is 30 years old and works as a Software Engineer.'

In [23]:
#create the tool for personal information
@tool
def tool_peroanl_info(query:str)->str:
    """Search personal information database and return
    a person's name, age, and location."""

    infos=[{
        "name":"Junaid",
        "age":"20",
        "location":"Germany"

    },
    {    "name":"bilal",
        "age":"22",
        "location":"Dutchland"
        }]
    
    for info in infos:
        if info["name"].lower()==query.lower():
            #return f"{info["name"]} is {info["age"]} and live in {info["location"]}"
            return f"{info['name']} is {info['age']} and lives in {info['location']}"
    return "no information"


In [24]:
tool_peroanl_info.invoke("bilal")

'bilal is 22 and lives in Dutchland'

In [25]:
#now tool is created now we will bind these tool in llm

In [26]:
llm=ChatOpenAI(model="gpt-5-mini")
llm
#llm is created

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10', 'langchain-openai': '1.3.2'}}, output_version=None, profile={'name': 'GPT-5 Mini', 'release_date': '2025-08-07', 'last_updated': '2025-08-07', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x117a606e0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x117a61160>, root_client=<openai.OpenAI object at 0x116e03a10

In [27]:
# to bind , we need bind tool kit

In [28]:
tool_kit=[

    tool_duck_duck_go_search,
    tool_wiki_pedia_search,
    tool_arxiv_search,
    tool_peroanl_info,
    tool_personal_info2 

]

In [29]:
tool_kit

[StructuredTool(name='tool_duck_duck_go_search', description='Use this tool when you need to answer questions about current events or general knowledge.', args_schema=<class 'langchain_core.utils.pydantic.tool_duck_duck_go_search'>, func=<function tool_duck_duck_go_search at 0x116e74ea0>),
 StructuredTool(name='tool_wiki_pedia_search', description='Use this tool when you need to answer questions about persons, places, etc.', args_schema=<class 'langchain_core.utils.pydantic.tool_wiki_pedia_search'>, func=<function tool_wiki_pedia_search at 0x116e75bc0>),
 StructuredTool(name='tool_arxiv_search', description='Use this tool when you need to answer questions about scientific papers or research topics.', args_schema=<class 'langchain_core.utils.pydantic.tool_arxiv_search'>, func=<function tool_arxiv_search at 0x1170b7f60>),
 StructuredTool(name='tool_peroanl_info', description="Search personal information database and return\na person's name, age, and location.", args_schema=<class 'langch

In [30]:
llm_bind=llm.bind_tools(tool_kit)
llm_bind
#. bind tool that allow to bind the tool with llm

_ChatModelBinding(bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10', 'langchain-openai': '1.3.2'}}, output_version=None, profile={'name': 'GPT-5 Mini', 'release_date': '2025-08-07', 'last_updated': '2025-08-07', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x117a606e0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x117a61160>, root_client=<openai.Open

In [32]:
#now i will call the llm and ask the age of bilal, it will pick the information but not final and give suggestion
#to solve this we need agent , it will works as loop and fectch the exact information
#this is reason and act, we use react agent class in llm

In [33]:
llm_bind.invoke("what the location of bilal")

AIMessage(content='I’m missing important details. Which “Bilal” do you mean (full name, any other identifying info)? Also: is this a public figure or a private person?\n\nImportant: I can’t help locate or track a private individual without their consent. If the person is private and you don’t have their permission, I can’t provide their current location. If it’s an emergency or you’re worried for their safety, contact local authorities right away.\n\nIf you clarify, I can help in one of these allowed ways:\n- If Bilal is a public figure, give me the full name or context (e.g., “Bilal the singer”), and I can look up publicly available location-related info.\n- If you have permission to find a private person, tell me what info you already have (full name, city, age, last known contacts), and I can suggest safe, legal search steps.\n- I can outline general methods you can use: Google search (name + city/keywords), LinkedIn, Facebook/Instagram/X, mutual contacts, email/phone lookup, public

In [34]:
llm_bind.invoke("What's the age of John Doe?. Make tool calls if necessary.")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 155, 'prompt_tokens': 320, 'total_tokens': 475, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dtd3brQTUh1YEejRnJYJPYCLiYZPk', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ef073-46a6-7bb1-8b1e-eb5481850238-0', tool_calls=[{'name': 'tool_personal_info2', 'args': {'name': 'John Doe'}, 'id': 'call_6ovOA6x6vFiplZva3xfAnKqc', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 320, 'output_tokens': 155, 'total_tokens': 475, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 128}})

In [35]:
llm_bind = llm.bind_tools(tool_kit)

response = llm_bind.invoke("what the location of bilal")
print(response)

content='Do you mean a specific person named Bilal? I need more detail to help — for example, is this a public figure (a singer, historical figure, etc.) or a private person you know?\n\nImportant: I can’t help locate or provide personal location information for private individuals without their consent. If you’re asking about a private person, I can instead suggest safe, legal ways to try to contact them (checking mutual contacts, social media profiles they’ve shared publicly, calling, or contacting authorities if they’re missing).\n\nIf you mean a public or historical Bilal, tell me which one (full name or context) and I can search public information or give background. If you want, give the full name and confirm you’re asking about publicly available info and I’ll look it up.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 488, 'prompt_tokens': 313, 'total_tokens': 801, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio